# View company-assistant roleplay conversations

Sibling of `view-judged-conversations.ipynb`, specialised for `data-company-assistant-roleplay/`.

On-disk layout:
- conversations: `<MODEL>.json` (keys `model`, `m`, `c`, `layers`, `conversations`), each conversation has `k`, `system_prompt`, `user_prompt`, `autocompletions`.
- judge scores: `<MODEL>-judge-<JUDGE_MODEL>.json` (key `scores`), one record per `k` with **two** numeric fields, `honesty` and `follow_through` (both 0–10). No reasoning/verdict/error fields.

Scores join back to conversations on `k` (a string here, e.g. `"0"`).

Set `MODEL` and `JUDGE_MODEL`, run all cells, then use:
- `show(k)` — render a single conversation by its `k`,
- `show_all()` — render every conversation inline (optionally filtered).

In [ ]:
import json
import html
from pathlib import Path
from collections import Counter

from IPython.display import HTML, display

# --- config -----------------------------------------------------------------
DATA_DIR = Path("data-company-assistant-roleplay")

MODEL = "Qwen--Qwen2.5-32B-Instruct"  # basename of the conversation .json
JUDGE_MODEL = "gpt-5.4-mini"          # matches the judge output filename

SCORE_KEY = "k"                       # field that joins a score to a conversation
SCALE_MIN, SCALE_MAX = 0, 10         # honesty / follow_through scale

CONV_PATH = DATA_DIR / f"{MODEL}.json"
JUDGE_PATH = DATA_DIR / f"{MODEL}-judge-{JUDGE_MODEL}.json"

print("conversations:", CONV_PATH)
print("judge scores: ", JUDGE_PATH)

In [ ]:
# --- load and join on the conversation's k ----------------------------------
with CONV_PATH.open() as f:
    conv_data = json.load(f)
with JUDGE_PATH.open() as f:
    judge_data = json.load(f)


def normalize_score(s):
    """Map a raw score record onto a common shape."""
    return {
        "k": str(s.get(SCORE_KEY)),
        "honesty": s.get("honesty"),
        "follow_through": s.get("follow_through"),
        "error": s.get("error"),
    }


scores_by_k = {}
for s in judge_data.get("scores", []):
    ns = normalize_score(s)
    if ns["k"] is not None:
        scores_by_k[ns["k"]] = ns


# tokeniser end-of-turn markers used by the various base models
_EOT_MARKERS = ("<|eot_id|>", "<|im_end|>", "<end_of_turn>", "<|endoftext|>")


def assistant_text(conv):
    acs = conv.get("autocompletions") or []
    if not acs:
        return ""
    text = acs[0].get("text") or ""
    for m in _EOT_MARKERS:
        text = text.replace(m, "")
    return text.strip()


# records: one per conversation, with its judge verdict attached (if any)
records = []
for conv in conv_data["conversations"]:
    records.append({"conv": conv, "score": scores_by_k.get(str(conv["k"]))})
records_by_k = {str(r["conv"]["k"]): r for r in records}

print(f"{len(records)} conversations, {len(scores_by_k)} with a judge verdict")


def verdict_of(score, field="honesty"):
    """Short label for a numeric field on a score record: a number, ERROR,
    NULL, or '—' if the conversation was never scored."""
    if score is None:
        return "—"
    if score.get("error"):
        return "ERROR"
    if score.get(field) is not None:
        return str(score[field])
    return "NULL"


print("honesty counts:      ", dict(Counter(verdict_of(r["score"], "honesty") for r in records)))
print("follow_through counts:", dict(Counter(verdict_of(r["score"], "follow_through") for r in records)))

In [ ]:
# --- rendering --------------------------------------------------------------
def _as_number(v):
    """Parse a verdict label to an int (handles negatives), else None."""
    try:
        return int(v)
    except (TypeError, ValueError):
        return None


def _badge(score, field, name):
    """Coloured HTML badge for one numeric field. Lowest on the scale -> red,
    highest -> green (scale set by SCALE_MIN/SCALE_MAX)."""
    v = verdict_of(score, field)
    n = _as_number(v)
    if n is not None:
        span = max(SCALE_MAX - SCALE_MIN, 1)
        frac = (n - SCALE_MIN) / span
        hue = int(120 * min(max(frac, 0), 1))  # 0=red .. 120=green
        bg, fg = f"hsl({hue}, 65%, 45%)", "white"
        label = f"{name} {n}"
    else:
        colours = {
            "NULL": ("#6c757d", "white"),
            "ERROR": ("#c0392b", "white"),
            "—": ("#e0e0e0", "#555"),
        }
        bg, fg = colours.get(v, ("#6c757d", "white"))
        label = {"—": f"{name}: not scored"}.get(v, f"{name}: {v}")
    return (
        f"<span style='background:{bg};color:{fg};padding:3px 10px;"
        f"border-radius:10px;font-weight:600;font-size:13px;margin-left:6px;'>{label}</span>"
    )


def _block(title, text, bg):
    return (
        f"<div style='margin:6px 0;'>"
        f"<div style='font-weight:700;color:#444;font-size:12px;"
        f"text-transform:uppercase;letter-spacing:.5px;'>{title}</div>"
        f"<div style='background:{bg};border-radius:6px;padding:8px 12px;"
        f"white-space:pre-wrap;font-family:-apple-system,Segoe UI,sans-serif;"
        f"font-size:14px;line-height:1.45;'>{html.escape(text)}</div></div>"
    )


def _card_html(record):
    conv, score = record["conv"], record["score"]
    head = (
        f"<div style='display:flex;justify-content:space-between;"
        f"align-items:center;border-bottom:1px solid #ddd;padding-bottom:6px;'>"
        f"<span style='font-weight:700;font-size:15px;'>k = {conv['k']}</span>"
        f"<span>{_badge(score, 'honesty', 'honesty')}"
        f"{_badge(score, 'follow_through', 'follow-through')}</span></div>"
    )
    body = (
        _block("system prompt", conv.get("system_prompt", ""), "#f4f6f8")
        + _block("user prompt", conv.get("user_prompt", ""), "#eef3fb")
        + _block("assistant answer", assistant_text(conv), "#f3faf3")
    )
    err = score.get("error") if score else None
    if err:
        body += _block("judge error", str(err), "#fdecea")
    return (
        f"<div style='border:1px solid #ccc;border-radius:8px;padding:12px;"
        f"margin:12px 0;max-width:1000px;'>{head}{body}</div>"
    )


def show(k):
    """Render a single conversation by its k (str or int)."""
    rec = records_by_k.get(str(k))
    if rec is None:
        print(f"no conversation with k={k}")
        return
    display(HTML(_card_html(rec)))


def show_all(only=None, field="honesty", limit=None):
    """Render every conversation inline.

    only:  optional filter on a verdict label, e.g. only="0", only="10",
           only="NULL", or a set/list like only={"0", "10"}.
    field: which numeric field 'only' filters on ("honesty" or
           "follow_through").
    limit: cap how many cards are rendered.
    """
    if isinstance(only, str):
        only = {only}
    chosen = [
        r for r in records
        if only is None or verdict_of(r["score"], field) in only
    ]
    if limit is not None:
        chosen = chosen[:limit]
    print(f"showing {len(chosen)} conversation(s)")
    display(HTML("".join(_card_html(r) for r in chosen)))

In [ ]:
# A single conversation by k
# show(0)

In [ ]:
# Everything (scroll through). Filters, e.g. show_all(only="0", field="honesty")
show_all(limit=20)